In [0]:
from pyspark.sql.functions import col, trim, lower, to_date
from delta.tables import DeltaTable

incremental_path = (
    "abfss://bronze@ecommercenidhi.dfs.core.windows.net/"
    "incremental/orders_incremental.csv"
)

incremental_orders = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(incremental_path)
    .dropDuplicates()
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("order_date", to_date(col("order_date")))
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("amount", col("amount").cast("double"))
    .withColumn("status", lower(trim(col("status"))))
)

display(incremental_orders)
print("Incremental records:", incremental_orders.count())

In [0]:
silver_orders_path = (
    "abfss://silver@ecommercenidhi.dfs.core.windows.net/orders"
)

silver_orders = DeltaTable.forPath(
    spark,
    silver_orders_path
)

(
    silver_orders.alias("target")
    .merge(
        incremental_orders.alias("source"),
        "target.order_id = source.order_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
silver_orders_check = (
    spark.read
    .format("delta")
    .load(silver_orders_path)
)

print("Silver orders after MERGE:", silver_orders_check.count())

In [0]:
display(
    silver_orders_check
    .orderBy(col("order_id").desc())
    .limit(25)
)

In [0]:
from pyspark.sql import Row

cdc_data = [
    Row(
        order_id="O0502",
        customer_id="C0042",
        order_date="2026-09-03",
        product_id="P017",
        quantity=4,
        amount=2396.0,
        status="cancelled"
    ),
    Row(
        order_id="O0522",
        customer_id="C0077",
        order_date="2026-09-10",
        product_id="P033",
        quantity=2,
        amount=998.0,
        status="completed"
    )
]

cdc_orders = spark.createDataFrame(cdc_data)

from pyspark.sql.functions import to_date, col, trim, lower

cdc_orders = (
    cdc_orders
    .withColumn("order_date", to_date(col("order_date")))
    .withColumn("status", lower(trim(col("status"))))
)

display(cdc_orders)

In [0]:
from delta.tables import DeltaTable

silver_orders = DeltaTable.forPath(
    spark,
    "abfss://silver@ecommercenidhi.dfs.core.windows.net/orders"
)

(
    silver_orders.alias("target")
    .merge(
        cdc_orders.alias("source"),
        "target.order_id = source.order_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
display(
    spark.read
    .format("delta")
    .load("abfss://silver@ecommercenidhi.dfs.core.windows.net/orders")
    .filter(col("order_id").isin("O0502", "O0522"))
)